# Split learning + Projected
### ResNet18 on CIFAR-100 + α = 0.1

In [1]:
#=============================================================================
# Split Learning + Orthogonal Projection (Gated) on CIFAR-100
# - Warmup + alpha ramp + norm-preserving projection
# - Train-side feature reservoir (no test leakage)
# ============================================================================
import torch
from torch import nn
from torchvision import transforms
from torchvision import datasets
from torch.utils.data import DataLoader, Dataset
import torch.nn.functional as F
import math, time
import os.path
import pandas as pd

from sklearn.model_selection import train_test_split
from PIL import Image
from glob import glob 
from pandas import DataFrame
import random
import numpy as np
import os


import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import copy


#=====================================================================================================
#                           Repro & Device
#=====================================================================================================

SEED = 1234
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed(SEED)
if torch.cuda.is_available():
    torch.backends.cudnn.deterministic = True
    print(torch.cuda.get_device_name(0))    
    

def _sync_cuda():
    if torch.cuda.is_available():
        torch.cuda.synchronize()
        
        
#=====================================================================================================
#                                   Parameters Change
#===================================================================================================== 
CUT_MODE = "mid"
'''
"early" = after layer3              --> [B, 64, 32, 32]
"mid"   = after layer5              --> [B, 256, 8, 8]
"late"  = after layer6 + avgpool    --> [B, 512, 1, 1]
'''

cut_at = {"early":"Early cut after layer3",
          "mid":"Mid cut after layer5",
          "late":"Late cut after layer6 + avgpool"}[CUT_MODE]



ALPHA = 0.1
DATASET_NAME = "CIFAR10"
DATA_ROOT = "data"
program = f"SL + Projected on {DATASET_NAME} - {ALPHA} - {CUT_MODE}"
print(f"---------{program}----------")              # this is to identify the program in the slurm outputs files



device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')


# To print in color -------test/train of the client side
def prRed(skk): print("\033[91m {}\033[00m" .format(skk)) 
def prGreen(skk): print("\033[92m {}\033[00m" .format(skk))


#=====================================================================================================
#                            SL / Training Config
#===================================================================================================== 
 
NUM_CLIENTS = 10
EPOCHS = 200           # global rounds
frac   = 1.0           # fraction of clients per round
lr_srv = 0.001         # server LR (SGD)
lr_cli = 0.001         # client LR (SGD)
weight_decay = 5e-4
momentum = 0.9
BATCH_SIZE = 256
local_ep = 5
NUM_WORKERS = 4

#=====================================================================================================
#                           Orthogonal Config
#=====================================================================================================
# Orthogonal projection config deque
ORTH = True
rebuild_every = 5         # rebuild subspace cadence (rounds)
WARMUP_ROUNDS = 10        # no projection before this
alpha_max = 0.1           # target projection strength
alpha_min = 0.0
alpha_warm_len = 10       # rounds to ramp alpha after warmup


# --- Cut-dependent OP buffer sizes ---
'''
early cut: D = 643232 = 65536 (big) --> change TOP_R to 4 so it's not huge
mid cut: D = 25688 = 16384
late cut: D = 51211 = 512'
'''
if CUT_MODE == "early":
    FEATURE_MEM_CAP = 512
    TOP_R = 4
elif CUT_MODE == "mid":
    FEATURE_MEM_CAP = 2048
    TOP_R = 8
else:  # late
    FEATURE_MEM_CAP = 4096
    TOP_R = 8
    

# total samples for building U_old
# Feature reservoir (train-side only)
FEATURE_MEM = []
_seen = 0

# holding D x r matrix of principal directions at the cut
U_old = None


# ====================== Confirm =======================================
prRed(f"\tData:\t\t{DATASET_NAME}\n\tCut:\t\t{cut_at}\n\tGlobal Rounds:\t{EPOCHS}\n\tClients:\t{NUM_CLIENTS}\n\tLocal Epochs:\t{local_ep}\n\tα:\t\t{ALPHA}\n\tWarm-up:\t{WARMUP_ROUNDS}\n\tRebuild:\t{rebuild_every}")


#=====================================================================================================
#                       Round 4 New: Projection Gate
#=====================================================================================================

def proj_gate_strength(round_idx: int):
    """Alpha schedule for projection."""
    if round_idx < WARMUP_ROUNDS:
        return 0.0
    t = min(1.0, (round_idx - WARMUP_ROUNDS) / max(1, alpha_warm_len))
    return alpha_min + t * (alpha_max - alpha_min)

#=====================================================================================================
#                           Projection Help
#=====================================================================================================
def project_off_subspace_gated(grad_batch: torch.Tensor, U: torch.Tensor, alpha: float, preserve_norm=True):
    """g' = g - alpha * P_U(g), optionally rescale to preserve per-sample norm."""
    if (U is None) or (alpha <= 0.0):
        return grad_batch
    B = grad_batch.shape[0]
    g = grad_batch.view(B, -1)              # [B, D]
    comp = (g @ U) @ U.T                    # [B, D]
    if preserve_norm:
        n0 = g.norm(dim=1, keepdim=True).clamp_min(1e-12)
    g2 = g - alpha * comp
    if preserve_norm:
        n1 = g2.norm(dim=1, keepdim=True).clamp_min(1e-12)
        g2 = g2 * (n0 / n1)
    return g2.view_as(grad_batch)

def add_to_reservoir(rows_cpu_fp16: torch.Tensor):
    """Standard reservoir sampling to keep a fixed-size unbiased buffer."""
    global _seen, FEATURE_MEM
    for r in rows_cpu_fp16:
        _seen += 1
        if len(FEATURE_MEM) < FEATURE_MEM_CAP:
            FEATURE_MEM.append(r)
        else:
            j = np.random.randint(0, _seen)
            if j < FEATURE_MEM_CAP:
                FEATURE_MEM[j] = r

def collect_cut_feats_from_train(fx_client: torch.Tensor):
    """Collect a small subset of cut features from TRAIN forward passes only."""
    with torch.no_grad():
        feats = fx_client.detach().view(fx_client.shape[0], -1)  # [B, D]
        k = min(24, feats.shape[0])
        idx = torch.randperm(feats.shape[0], device=feats.device)[:k]
        add_to_reservoir(feats[idx].cpu().to(torch.float16))


#=====================================================================================================
#         Subspace builder (simple PCA/SVD on buffered features)
#=====================================================================================================
def build_U_old_from_memory(top_r, device='cpu'):
    if len(FEATURE_MEM) == 0:
        return None
    X = torch.stack(FEATURE_MEM).to(torch.float32)  # [N, D]
    X = X - X.mean(dim=0, keepdim=True)
    # q controls rank in pca_lowrank
    q = min(top_r, X.shape[1], X.shape[0]-1) if X.shape[0] > 1 else 1
    U, S, V = torch.pca_lowrank(X.cpu(), q=q, center=False)
    return V[:, :top_r].to(device, dtype=torch.float32)


def clear_feature_mem():
    global FEATURE_MEM, _seen
    FEATURE_MEM.clear()
    _seen = 0


    
#=====================================================================================================
#                                          DATA Loader
#=====================================================================================================

CIFAR_MEAN = {
    "CIFAR10": [0.4914, 0.4822, 0.4465],
    "CIFAR100": [0.5071, 0.4867, 0.4408],
}[DATASET_NAME]
CIFAR_STD = {
    "CIFAR10": [0.2470, 0.2435, 0.2616],
    "CIFAR100": [0.2675, 0.2565, 0.2761],
}[DATASET_NAME]



# RandAugment (fallback if old torchvision)
try:
    from torchvision.transforms import RandAugment
    ra = [RandAugment(num_ops=2, magnitude=9)]
except Exception:
    ra = []

train_transforms = transforms.Compose([
    transforms.RandomHorizontalFlip(),
    transforms.RandomCrop(32, padding=4),
] + ra + [
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
])

test_transforms = transforms.Compose([
    transforms.ToTensor(),
    transforms.Normalize(mean=CIFAR_MEAN, std=CIFAR_STD),
])



if DATASET_NAME == "CIFAR10":
    dataset_train = datasets.CIFAR10(
        root=DATA_ROOT, train=True, download=True, transform=train_transforms
    )
    dataset_test = datasets.CIFAR10(
        root=DATA_ROOT, train=False, download=True, transform=test_transforms
    )
    NUM_CLASSES = 10
else:
    dataset_train = datasets.CIFAR100(
        root=DATA_ROOT, train=True, download=True, transform=train_transforms
    )
    dataset_test = datasets.CIFAR100(
        root=DATA_ROOT, train=False, download=True, transform=test_transforms
    )
    NUM_CLASSES = 100

#=============================================================================
#                         Forgetting metric setup 
#============================================================================= 

from collections import defaultdict

T = EPOCHS
class DatasetSplit(Dataset):  # (you already have this in both scripts; keep just one copy)
    def __init__(self, dataset, idxs):
        self.dataset, self.idxs = dataset, list(idxs)
    def __len__(self): return len(self.idxs)
    def __getitem__(self, i): return self.dataset[self.idxs[i]]

val_loaders = {}
# Deterministic, disjoint slices of the test set: window t gets items t, t+T, t+2T, ...
rng_idx = np.arange(len(dataset_test))
for t in range(T):
    idxs = rng_idx[t::T]
    val_loaders[t] = DataLoader(DatasetSplit(dataset_test, idxs),
                                batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=NUM_WORKERS)

best_loss_per_window = defaultdict(lambda: float('inf'))  # stores min historical loss per window
forgetting_log = []  # per round dicts: {"round": t, "mean_forgetting": x, "per_window": {...}}
# ============================================================================

def _eval_loss_on_loader(model_client, model_server, loader, device, criterion):
    """Evaluate mean loss on a loader using split models."""
    # Handle DataParallel transparently
    mc = model_client.module if isinstance(model_client, nn.DataParallel) else model_client
    ms = model_server.module if isinstance(model_server, nn.DataParallel) else model_server
    mc.eval(); ms.eval()
    loss_sum, n = 0.0, 0
    with torch.no_grad():
        for images, labels in loader:
            images, labels = images.to(device), labels.to(device)
            fx = mc(images)
            logits = ms(fx)
            loss = criterion(logits, labels)
            bs = labels.size(0)
            loss_sum += loss.item() * bs
            n += bs
    return loss_sum / max(1, n)

#=====================================================================================================
#                           IID / Dirichlet partitioning
#=====================================================================================================

def dataset_iid(dataset, num_users):
    num_items = int(len(dataset) / num_users)
    dict_users, all_idxs = {}, [i for i in range(len(dataset))]
    for i in range(num_users):
        dict_users[i] = set(np.random.choice(all_idxs, num_items, replace=False))
        all_idxs = list(set(all_idxs) - dict_users[i])
    return dict_users

def dataset_noniid_dirichlet(dataset, num_users, alpha=ALPHA, seed=SEED):
    rng = np.random.default_rng(seed)
    labels = np.array(dataset.targets)
    n_classes = len(np.unique(labels))
    dict_users = {i: set() for i in range(num_users)}
    for c in range(n_classes):
        idxs = np.where(labels == c)[0]
        rng.shuffle(idxs)
        p = rng.dirichlet([alpha] * num_users)
        cuts = (np.cumsum(p) * len(idxs)).astype(int)[:-1]
        splits = np.split(idxs, cuts)
        for client_id, split in enumerate(splits):
            dict_users[client_id].update(split.tolist())
    return dict_users

dict_users      = dataset_noniid_dirichlet(dataset_train, NUM_CLIENTS, alpha=ALPHA)
dict_users_test = dataset_iid(dataset_test, NUM_CLIENTS)

 
           
#=====================================================================================================
#                           Model definition
#=====================================================================================================
# Model at server side
class Baseblock(nn.Module):
    expansion = 1
    def __init__(self, in_planes, planes, stride=1, dim_change=None):
        super().__init__()
        self.conv1 = nn.Conv2d(in_planes, planes, kernel_size=3, stride=stride, padding=1)
        self.bn1   = nn.BatchNorm2d(planes)
        self.conv2 = nn.Conv2d(planes, planes, kernel_size=3, stride=1, padding=1)
        self.bn2   = nn.BatchNorm2d(planes)
        self.dim_change = dim_change
    def forward(self, x):
        res = x
        out = F.relu(self.bn1(self.conv1(x)))
        out = self.bn2(self.conv2(out))
        if self.dim_change is not None:
            res = self.dim_change(res)
        out += res
        out = F.relu(out)
        return out
    
    
class SplitClient(nn.Module):
    def __init__(self, cut_mode: str):
        super().__init__()
        self.cut_mode = cut_mode

        # stem + layer2 (your current client)
        self.layer1 = nn.Sequential(
            nn.Conv2d(3, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
        )
        self.layer2 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
        )

        # layer3 (currently on server)
        self.layer3 = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
            nn.ReLU(inplace=True),
            nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1, bias=False),
            nn.BatchNorm2d(64),
        )

        # layer4/5/6 (same construction logic as your server)
        self.input_planes = 64
        self.layer4 = self._layer(Baseblock, 128, 2, stride=2)
        self.layer5 = self._layer(Baseblock, 256, 2, stride=2)
        self.layer6 = self._layer(Baseblock, 512, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

    def _layer(self, block, planes, num_layers, stride=2):
        dim_change = None
        if stride != 1 or planes != self.input_planes * block.expansion:
            dim_change = nn.Sequential(
                nn.Conv2d(self.input_planes, planes * block.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion),
            )
        layers = [block(self.input_planes, planes, stride=stride, dim_change=dim_change)]
        self.input_planes = planes * block.expansion
        for _ in range(1, num_layers):
            layers.append(block(self.input_planes, planes))
        return nn.Sequential(*layers)

    def forward(self, x):
        x0 = self.layer1(x)
        x1 = self.layer2(x0)
        x = F.relu(x1 + x0)

        # layer3 residual
        x2 = self.layer3(x)
        x = F.relu(x2 + x)

        if self.cut_mode == "early":
            return x                      # after layer3  -> [B,64,32,32]

        x = self.layer4(x)
        x = self.layer5(x)

        if self.cut_mode == "mid":
            return x                      # after layer5  -> [B,256,8,8]

        x = self.layer6(x)
        x = self.avgpool(x)

        if self.cut_mode == "late":
            return x                      # after layer6+avgpool -> [B,512,1,1]

        raise ValueError("Unknown cut_mode")


class SplitServer(nn.Module):
    def __init__(self, cut_mode: str, num_classes: int):
        super().__init__()
        self.cut_mode = cut_mode

        # If cut is early, server needs layer4/5/6 + avgpool + fc
        self.input_planes = 64
        self.layer4 = self._layer(Baseblock, 128, 2, stride=2)
        self.layer5 = self._layer(Baseblock, 256, 2, stride=2)
        self.layer6 = self._layer(Baseblock, 512, 2, stride=2)
        self.avgpool = nn.AdaptiveAvgPool2d((1, 1))

        self.fc = nn.Linear(512, num_classes)

    def _layer(self, block, planes, num_layers, stride=2):
        dim_change = None
        if stride != 1 or planes != self.input_planes * block.expansion:
            dim_change = nn.Sequential(
                nn.Conv2d(self.input_planes, planes * block.expansion, kernel_size=1, stride=stride, bias=False),
                nn.BatchNorm2d(planes * block.expansion),
            )
        layers = [block(self.input_planes, planes, stride=stride, dim_change=dim_change)]
        self.input_planes = planes * block.expansion
        for _ in range(1, num_layers):
            layers.append(block(self.input_planes, planes))
        return nn.Sequential(*layers)

    def forward(self, fx):
        # fx shape depends on cut
        if self.cut_mode == "early":
            x = self.layer4(fx)
            x = self.layer5(x)
            x = self.layer6(x)
            x = self.avgpool(x)
            x = x.view(x.size(0), -1)      # [B,512]
            return self.fc(x)

        if self.cut_mode == "mid":
            x = self.layer6(fx)
            x = self.avgpool(x)
            x = x.view(x.size(0), -1)      # [B,512]
            return self.fc(x)

        if self.cut_mode == "late":
            x = fx.view(fx.size(0), -1)    # [B,512]
            return self.fc(x)

        raise ValueError("Unknown cut_mode")


net_glob_client = SplitClient(CUT_MODE).to(device)
net_glob_server = SplitServer(CUT_MODE, NUM_CLASSES).to(device)

if torch.cuda.device_count() > 1:
    net_glob_client = nn.DataParallel(net_glob_client)
    net_glob_server = nn.DataParallel(net_glob_server)

with torch.no_grad():
    x = torch.randn(2, 3, 32, 32).to(device)
    fx = net_glob_client(x)
    print("CUT_MODE:", CUT_MODE, "| fx shape:", tuple(fx.shape))

#===================================================================================
# For Server Side Loss and Accuracy 
loss_train_collect = []
acc_train_collect = []
loss_test_collect = []
acc_test_collect = []
batch_acc_train = []
batch_loss_train = []
batch_acc_test = []
batch_loss_test = []
round_time_collect = [] # Stores seconds per global round

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)

#====================================================================================================
#                                  Server Side Program
#====================================================================================================
def calculate_accuracy(fx, y):
    preds = fx.max(1, keepdim=True)[1]
    correct = preds.eq(y.view_as(preds)).sum()
    acc = 100.0 * correct.float() / preds.shape[0]
    return acc

# to print train - test together in each round-- these are made global
acc_avg_all_user_train = 0
loss_avg_all_user_train = 0
loss_train_collect_user = []
acc_train_collect_user = []
loss_test_collect_user = []
acc_test_collect_user = []


#client idx collector
idx_collect = []
l_epoch_check = False
fed_check = False
count1 = 0
count2 = 0
ell_global = 0  # set at each global round


# Persistent server optimizer + scheduler
optimizer_server = torch.optim.SGD(net_glob_server.parameters(), lr=lr_srv, momentum=momentum, weight_decay=weight_decay)
server_sched = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer_server, T_max=EPOCHS, eta_min=5e-4)



#====================================================================================================
#                                  Server Routines
#====================================================================================================

# Server-side function associated with Training 
def train_server(fx_client, y, l_epoch_count, l_epoch, idx, len_batch):
    global net_glob_server, optimizer_server, criterion
    global batch_acc_train, batch_loss_train, l_epoch_check, fed_check
    global loss_train_collect, acc_train_collect, count1
    global acc_avg_all_user_train, loss_avg_all_user_train, idx_collect
    global loss_train_collect_user, acc_train_collect_user, U_old, ell_global

    # train and update
    net_glob_server.train()
    optimizer_server.zero_grad()

    #---------forward prop-------------
    # make fx_client a LEAF so .grad is populated
    fx_client = fx_client.to(device).detach().requires_grad_(True)
    y = y.to(device)

    logits = net_glob_server(fx_client)
    loss = criterion(logits, y)
    acc  = calculate_accuracy(logits, y)

    #--------backward prop--------------
    loss.backward()

    # -------------- ORTH PROJECTION AT THE CUT GRADIENT --------------
    # OP at the cut
    # gradient w.r.t. cut activations (now guaranteed)
    dfx_client = fx_client.grad.detach()

    # Orthogonal projection at cut gradient (gated)
    if ORTH and (U_old is not None):
        alpha = proj_gate_strength(ell_global)
        dfx_client = project_off_subspace_gated(
            dfx_client, U_old.to(dfx_client.dtype), alpha, preserve_norm=True
        )

    optimizer_server.step()

    # bookkeeping
    batch_loss_train.append(loss.item())
    batch_acc_train.append(acc.item())

    count1 += 1
    if count1 == len_batch:
        acc_avg_train  = sum(batch_acc_train) / len(batch_acc_train)
        loss_avg_train = sum(batch_loss_train) / len(batch_loss_train)
        batch_acc_train.clear()
        batch_loss_train.clear()
        count1 = 0

        prRed(f'Client{idx} Train => Local Epoch: {l_epoch_count} \tAcc: {acc_avg_train:.3f} \tLoss: {loss_avg_train:.4f}')

        if l_epoch_count == l_epoch - 1:
            l_epoch_check = True
            loss_train_collect_user.append(loss_avg_train)
            acc_train_collect_user.append(acc_avg_train)
            if idx not in idx_collect:
                idx_collect.append(idx)

        if len(idx_collect) == NUM_CLIENTS:
            fed_check = True
            idx_collect.clear()
            acc_avg_all_user_train  = sum(acc_train_collect_user) / len(acc_train_collect_user)
            loss_avg_all_user_train = sum(loss_train_collect_user) / len(loss_train_collect_user)
            loss_train_collect.append(loss_avg_all_user_train)
            acc_train_collect.append(acc_avg_all_user_train)
            acc_train_collect_user.clear()
            loss_train_collect_user.clear()

    return dfx_client


def evaluate_server(fx_client, y, idx, len_batch, ell):
    global net_glob_server, criterion
    global batch_acc_test, batch_loss_test
    global loss_test_collect, acc_test_collect, count2
    global acc_avg_all_user_train, loss_avg_all_user_train, l_epoch_check, fed_check
    global loss_test_collect_user, acc_test_collect_user
    global U_old

    net_glob_server.eval()
    with torch.no_grad():
        fx_client = fx_client.to(device)
        y = y.to(device)
        fx_server = net_glob_server(fx_client)
        loss = criterion(fx_server, y)
        acc  = calculate_accuracy(fx_server, y)

        batch_loss_test.append(loss.item())
        batch_acc_test.append(acc.item())

        count2 += 1
        if count2 == len_batch:
            acc_avg_test = sum(batch_acc_test) / len(batch_acc_test)
            loss_avg_test = sum(batch_loss_test) / len(batch_loss_test)
            batch_acc_test.clear()
            batch_loss_test.clear()
            count2 = 0
            print(f"\033[92m Client{idx} Test  => \tAcc: {acc_avg_test:.3f} \tLoss: {loss_avg_test:.4f}\033[00m")

            if l_epoch_check:
                l_epoch_check = False
                loss_test_collect_user.append(loss_avg_test)
                acc_test_collect_user.append(acc_avg_test)

            if fed_check:
                fed_check = False
                acc_avg_all_user = sum(acc_test_collect_user) / len(acc_test_collect_user)
                loss_avg_all_user = sum(loss_test_collect_user) / len(loss_test_collect_user)
                loss_test_collect.append(loss_avg_all_user)
                acc_test_collect.append(acc_avg_all_user)
                acc_test_collect_user.clear()
                loss_test_collect_user.clear()

                print("====================== SERVER (OP-gated) ==========================")
                print(f" Train: Round {ell:3d}, Avg Acc {acc_avg_all_user_train:.3f} | Avg Loss {loss_avg_all_user_train:.3f}")
                print(f" Test : Round {ell:3d}, Avg Acc {acc_avg_all_user:.3f} | Avg Loss {loss_avg_all_user:.3f}")
                

            # Rebuild U on cadence (after warmup), and only if we have enough samples
            if ORTH and (ell >= WARMUP_ROUNDS) and (ell % rebuild_every == 0) and (len(FEATURE_MEM) >= 512):
                U_old = build_U_old_from_memory(top_r=TOP_R, device=device)
                # Reset reservoir to favor recent distribution   
                clear_feature_mem()

# ------------------------------ Client Wrapper -------------------------------
class Client(object):
    def __init__(self, net_client_model, idx, lr, device, dataset_train=None, dataset_test=None, idxs=None, idxs_test=None):
        self.idx = idx
        self.device = device
        self.lr = lr
        self.local_ep = local_ep
        self.ldr_train = DataLoader(DatasetSplit(dataset_train, idxs), batch_size=BATCH_SIZE, shuffle=True, drop_last=False, num_workers=NUM_WORKERS)
        self.ldr_test  = DataLoader(DatasetSplit(dataset_test,  idxs_test), batch_size=BATCH_SIZE, shuffle=False, drop_last=False, num_workers=NUM_WORKERS)

    def train(self, net):
        net.train()
        optimizer_client = torch.optim.SGD(net.parameters(), lr=self.lr, momentum=momentum, weight_decay=weight_decay)
        
        
        for ep in range(self.local_ep):
            len_batch = len(self.ldr_train)
            for batch_idx, (images, labels) in enumerate(self.ldr_train):
                images, labels = images.to(self.device), labels.to(self.device)
                optimizer_client.zero_grad()
                # Forward through client
                fx = net(images)
                # Collect train-side features for U reservoir
                if ORTH:
                    collect_cut_feats_from_train(fx)
                client_fx = fx.clone().detach().requires_grad_(True)

                # Send to server => receive gradient wrt cut activations
                dfx = train_server(client_fx, labels, ep, self.local_ep, self.idx, len_batch)

                # Backprop to client
                fx.backward(dfx)
                optimizer_client.step()
        return net.state_dict()

    def evaluate(self, net, ell):
        net.eval()
        # do NOT collect features here (no test leakage)
        with torch.no_grad():
            len_batch = len(self.ldr_test)
            for batch_idx, (images, labels) in enumerate(self.ldr_test):
                images, labels = images.to(self.device), labels.to(self.device)
                fx = net(images)
                # NO collect_cut_feats_from_train(fx) here.
                evaluate_server(fx, labels, self.idx, len_batch, ell)
        return

# ------------------------------ BN Freeze Helper -----------------------------
def freeze_client_bn(net):
    """Freeze client BN stats after warmup to stabilize cut gradients."""
    for m in net.modules():
        if isinstance(m, nn.BatchNorm2d):
            m.eval()
            

# ----------------------------- FedAvg helper ---------------------------------
def FedAvg(w_list):
    w_avg = copy.deepcopy(w_list[0])
    for k in w_avg.keys():
        for i in range(1, len(w_list)):
            w_avg[k] += w_list[i][k]
        w_avg[k] = torch.div(w_avg[k], len(w_list))
    return w_avg


# --------------------------------- Training ----------------------------------
acc_train_collect.clear(); loss_train_collect.clear()
acc_test_collect.clear();  loss_test_collect.clear()
round_time_collect.clear()
forgetting_log.clear()

pin = torch.cuda.is_available()

for iter in range(EPOCHS):
    ell_global = iter  # used by proj_gate_strength

    # freeze client BN after warmup
    if iter == WARMUP_ROUNDS:
        if isinstance(net_glob_client, nn.DataParallel):
            freeze_client_bn(net_glob_client.module)
        else:
            freeze_client_bn(net_glob_client)

    _sync_cuda()
    t0 = time.perf_counter()

    m = max(int(frac * NUM_CLIENTS), 1)
    idxs_users = np.random.choice(range(NUM_CLIENTS), m, replace=False)

    w_locals = []
    for idx in idxs_users:
        local = Client(
            net_glob_client, idx, lr_cli, device,
            dataset_train=dataset_train, dataset_test=dataset_test,
            idxs=dict_users[idx], idxs_test=dict_users_test[idx]
        )

        local_model = copy.deepcopy(net_glob_client).to(device)

        # train local copy starting from current global
        w_client = local.train(net=local_model)
        w_locals.append(copy.deepcopy(w_client))

        # evaluate the same trained local model
        local.evaluate(net=local_model, ell=iter)

    # ----- FedAvg: update global once per round -----
    w_glob = FedAvg(w_locals)
    net_glob_client.load_state_dict(w_glob)

    _sync_cuda()
    dt = time.perf_counter() - t0
    round_time_collect.append(dt)

    # ---- Forgetting update (KEEP if you want, but it's expensive) ----
    t = iter
    L_cur_t = _eval_loss_on_loader(net_glob_client, net_glob_server, val_loaders[t], device, criterion)
    best_loss_per_window[t] = min(best_loss_per_window[t], L_cur_t)

    per_win_fgt = {}
    for t_prev in range(t):
        L_now = _eval_loss_on_loader(net_glob_client, net_glob_server, val_loaders[t_prev], device, criterion)
        per_win_fgt[t_prev] = (L_now - best_loss_per_window[t_prev])

    mean_fgt = float(np.mean(list(per_win_fgt.values()))) if per_win_fgt else 0.0
    forgetting_log.append({"round": t, "mean_forgetting": mean_fgt, "per_window": per_win_fgt})
    print(f"Forgetting : Round {t}, mean_forgetting : {mean_fgt:.3f}")
    print("===================================================================")

    server_sched.step()

print("Training and Evaluation completed!")  

#===============================================================================
# Save output data to .excel file (we use for comparision plots)
#================ Save outputs ================#
# Align lengths safely (in case of any mismatch)
n = min(len(acc_train_collect), len(acc_test_collect), len(round_time_collect), len(forgetting_log))
round_process = list(range(1, n + 1))
mean_fgt_series = [d["mean_forgetting"] for d in forgetting_log[:n]]

df = pd.DataFrame({
    'round': round_process,
    'acc_train': acc_train_collect[:n],
    'acc_test':  acc_test_collect[:n],
    'round_time_sec': round_time_collect[:n],
    'mean_FgT_loss': mean_fgt_series,
    'projection': [True]*n,
    'top_r': [TOP_R]*n,
    'feature_mem_cap': [FEATURE_MEM_CAP]*n,
    'alpha': [ALPHA]*n,
    'num_users': [NUM_CLIENTS]*n,
    'lr_server': [lr_srv]*n,
    'lr_client': [lr_cli]*n,
    'seed': [SEED]*n,
})


file_name_csv  = program + ".csv"
df.to_csv(file_name_csv, index=False)
print("Saved:", file_name_csv)


#=============================================================================
#                         Program Completed
#============================================================================= 

NVIDIA A100-PCIE-40GB
---------SL + Projected on CIFAR-100 α=0.1----------
 	Data:		CIFAR100
	Cut:		Late-cut at Layer 6 + AvgPool
	Global Rounds:	100
	Clients:	10
	Local Epochs:	5
	α:		0.1
	Warm-up:	5
	Rebuild:	3


NameError: name 'DATA_ROOT' is not defined